In [0]:
# Log loss of recent predictions, published as the chart on the public prediction page.

import io
import os
import sys
from datetime import datetime, timedelta, timezone

import boto3
import matplotlib.pyplot as plt
from mlflow import MlflowClient
from pyspark.sql import functions as F

sys.path.append(os.path.abspath("../free_edition_models"))   # station_features.py is in free_edition_models
from station_features import build_predictors, build_targets

# Scheduled runs pass run_mode = "production" and publish the chart. Interactive runs only draw it.
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
IS_PRODUCTION = dbutils.widgets.get("run_mode") == "production"

# The defaults are the shared workspace's tables
dbutils.widgets.text("schema", "citibike_project.citibike")
dbutils.widgets.text("status_table", "citibike_project.citibike.silver_station_status")
dbutils.widgets.text("info_table", "citibike_project.citibike.silver_station_info")
dbutils.widgets.text("lookback_hours", "24")
SCHEMA = dbutils.widgets.get("schema")
STATUS_TABLE = dbutils.widgets.get("status_table")
INFO_TABLE = dbutils.widgets.get("info_table")
LOOKBACK_HOURS = int(dbutils.widgets.get("lookback_hours"))

PREDICTIONS_TABLE = f"{SCHEMA}.station_predictions"
MODEL_NAMES = {"bike": f"{SCHEMA}.bike_stockout_lr", "dock": f"{SCHEMA}.dock_stockout_lr"}
LOCAL_TZ = "America/New_York"
HORIZON_MIN = 30
MAX_OBSERVATION_GAP_MIN = 10

BUCKET_NAME = "cuny-sps-citibike-archive"
CHART_KEY = "exports/live/performance.png"

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
window_end = datetime.now(timezone.utc).replace(tzinfo=None)
window_start = window_end - timedelta(hours=LOOKBACK_HOURS)

# We calculate the outcomes, starting at one hour before the window start
recent_status = spark.table(STATUS_TABLE).filter(F.col("fetched_at") >= F.lit(window_start - timedelta(hours=1)))
station_info = spark.table(INFO_TABLE)
realized_outcomes = build_targets(
    build_predictors(recent_status, station_info, local_timezone=LOCAL_TZ, max_gap_minutes=MAX_OBSERVATION_GAP_MIN),
    horizon_minutes=HORIZON_MIN, max_gap_minutes=MAX_OBSERVATION_GAP_MIN,
)

# We join predictions and outcomes

predictions_with_outcomes = (
    spark.table(PREDICTIONS_TABLE)
    .filter(F.col("prediction_time") >= F.lit(window_start))
    .join(realized_outcomes.select("station_id", "fetched_at", "bike_target", "dock_target"),
          ["station_id", "fetched_at"], "inner")
)

prediction_hour = F.date_trunc("hour", F.from_utc_timestamp("prediction_time", LOCAL_TZ)).alias("prediction_hour")

bike_evaluations = (
    predictions_with_outcomes
    .filter((F.col("bike_state") == "eligible") & F.col("bike_target").isNotNull())
    .select(F.lit("bike").alias("model_type"), prediction_hour,
            F.col("p_bike_stockout").alias("predicted_probability"),
            F.col("bike_target").alias("observed"))
)
dock_evaluations = (
    predictions_with_outcomes
    .filter((F.col("dock_state") == "eligible") & F.col("dock_target").isNotNull())
    .select(F.lit("dock").alias("model_type"), prediction_hour,
            F.col("p_dock_stockout").alias("predicted_probability"),
            F.col("dock_target").alias("observed"))
)

# Protection against roundoff error 

clipped_probability = F.greatest(F.lit(1e-12), F.least(F.lit(1 - 1e-12), F.col("predicted_probability")))
log_loss = -(F.col("observed") * F.log(clipped_probability)
             + (1 - F.col("observed")) * F.log(1 - clipped_probability))

hourly_log_loss = (
    bike_evaluations.unionByName(dock_evaluations)
    .groupBy("model_type", "prediction_hour")
    .agg(F.avg(log_loss).alias("log_loss"))
    .orderBy("prediction_hour")
    .toPandas()
)

In [0]:
# We will plot the CV loss of the Champion model

client = MlflowClient(registry_uri="databricks-uc")

with plt.rc_context({"font.size": 15, "axes.titlesize": 18, "axes.labelsize": 15,
                     "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 13}):
    figure, axes = plt.subplots(1, 2, figsize=(16, 4.5), constrained_layout=True)

    for axis, model_type, label in zip(axes, ["bike", "dock"], ["Stockout", "Dockout"]):
        model_type_hours = hourly_log_loss[hourly_log_loss.model_type == model_type]

        champion = client.get_model_version_by_alias(MODEL_NAMES[model_type], "champion")
        training_log_loss = client.get_run(champion.run_id).data.metrics.get("cv_log_loss")

        axis.plot(model_type_hours.prediction_hour, model_type_hours.log_loss, "-", marker="o", linewidth=2.5, label="Hourly")
        if training_log_loss is not None:
            axis.axhline(training_log_loss, color="gray", linestyle="--", linewidth=2,
                         label=f"Training (version {champion.version}): {training_log_loss:.3f}")
            # I'm not sure the best way to select this scale
            axis.set_ylim(0, 3 * training_log_loss)
        axis.set(title=f"{label}: log loss by hour", ylabel="Log loss")
        axis.legend(loc="upper left")
        axis.tick_params(axis="x", labelrotation=45)

if IS_PRODUCTION:
    chart_buffer = io.BytesIO()
    figure.savefig(chart_buffer, format="png", dpi=110)
    s3 = boto3.client(
        "s3",
        aws_access_key_id=dbutils.secrets.get("citibike-export", "aws-access-key-id"),
        aws_secret_access_key=dbutils.secrets.get("citibike-export", "aws-secret-access-key"),
        region_name="us-west-2",
    )
    s3.put_object(Bucket=BUCKET_NAME, Key=CHART_KEY, Body=chart_buffer.getvalue(),
                  ContentType="image/png", CacheControl="max-age=60")
    print(f"Published https://{BUCKET_NAME}.s3.us-west-2.amazonaws.com/{CHART_KEY}")

plt.show()